In [0]:
%python
dbutils.widgets.removeAll()

In [0]:
create widget text storageName default "adlsproyect0408";

In [0]:
%python
storageName = dbutils.widgets.get("storageName")

In [0]:
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-metastore`
URL 'abfss://metastore@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas raw del Data Lake';

In [0]:
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-raw`
URL 'abfss://raw@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas raw del Data Lake';

In [0]:
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-bronze`
URL 'abfss://bronze@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas bronze del Data Lake';

In [0]:
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-silver`
URL 'abfss://silver@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas silver del Data Lake';

In [0]:
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-golden`
URL 'abfss://golden@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas golden del Data Lake';

In [0]:
DROP CATALOG IF EXISTS catalog_au CASCADE;

In [0]:
CREATE CATALOG IF NOT EXISTS catalog_au
MANAGED LOCATION 'abfss://metastore@${storageName}.dfs.core.windows.net/'
COMMENT 'Catalogo para la arquitectura medallion del ambiente de dev';

In [0]:
DROP SCHEMA IF EXISTS catalog_au.raw;
DROP SCHEMA IF EXISTS catalog_au.bronze;
DROP SCHEMA IF EXISTS catalog_au.silver;
DROP SCHEMA IF EXISTS catalog_au.golden;

In [0]:
%python
dbutils.fs.rm(f"abfss://bronze@{storageName}.dfs.core.windows.net/",True)
dbutils.fs.rm(f"abfss://silver@{storageName}.dfs.core.windows.net/",True)
dbutils.fs.rm(f"abfss://golden@{storageName}.dfs.core.windows.net/",True)

True

In [0]:
CREATE SCHEMA IF NOT EXISTS catalog_au.raw;
CREATE SCHEMA IF NOT EXISTS catalog_au.bronze;
CREATE SCHEMA IF NOT EXISTS catalog_au.silver;
CREATE SCHEMA IF NOT EXISTS catalog_au.golden;

###Tablas Bronze

In [0]:
CREATE TABLE IF NOT EXISTS catalog_au.bronze.spotify_tracks (
    `Unnamed_0` INT,
    track_id STRING,
    artists STRING,
    album_name STRING,
    track_name STRING,
    popularity INT,
    duration_ms INT,
    explicit BOOLEAN,
    danceability DOUBLE,
    energy DOUBLE,
    key INT,
    loudness DOUBLE,
    mode INT,
    speechiness DOUBLE,
    acousticness DOUBLE,
    instrumentalness DOUBLE,
    liveness DOUBLE,
    valence DOUBLE,
    tempo DOUBLE,
    time_signature INT,
    track_genre STRING,
    ingestion_date TIMESTAMP
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/spotify_tracks"

In [0]:
CREATE TABLE IF NOT EXISTS catalog_au.bronze.spotify_reviews (
    Time_submitted STRING,
    Review STRING,
    Rating INT,
    Total_thumbsup INT,
    Reply STRING,
    ingestion_date TIMESTAMP
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/spotify_reviews"

In [0]:
CREATE TABLE IF NOT EXISTS catalog_au.bronze.itunes_trends (
    itunes_track_id LONG,
    track_name STRING,
    artist_name STRING,
    genre STRING,
    price_usd DOUBLE,
    release_date STRING,
    ingestion_date TIMESTAMP
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/itunes_trends"

###Tablas Silver

In [0]:
CREATE TABLE IF NOT EXISTS catalog_au.silver.spotify_reviews_clean (
    Time_submitted TIMESTAMP,
    Review STRING,
    Rating INT,
    Total_thumbsup INT,
    Reply STRING,
    sentiment_category STRING, 
    ingestion_date TIMESTAMP,
    processed_date TIMESTAMP
)
USING DELTA
LOCATION "abfss://silver@${storageName}.dfs.core.windows.net/spotify_reviews_clean";

In [0]:
CREATE TABLE IF NOT EXISTS catalog_au.silver.music_trends_transformed (
    track_id STRING,
    track_name STRING,
    artists STRING,
    popularity INT,
    duration_ms INT,
    explicit BOOLEAN,
    popularity_category STRING,
    itunes_genre STRING,
    price_usd DOUBLE,
    is_hit STRING,
    ingestion_date TIMESTAMP
)
USING DELTA
LOCATION "abfss://silver@${storageName}.dfs.core.windows.net/music_trends_transformed";

###Tablas Golden

In [0]:
CREATE TABLE IF NOT EXISTS catalog_au.golden.gold_music_metrics (
    itunes_genre STRING,
    total_tracks LONG,
    avg_popularity DOUBLE,
    max_price_usd DOUBLE,
    total_hits LONG
)
USING DELTA
LOCATION "abfss://golden@${storageName}.dfs.core.windows.net/gold_music_metrics";



In [0]:
CREATE TABLE IF NOT EXISTS catalog_au.golden.gold_reviews_metrics (
    sentiment_category STRING,
    total_reviews LONG,
    avg_rating DOUBLE,
    total_thumbsup LONG
)
USING DELTA
LOCATION "abfss://golden@${storageName}.dfs.core.windows.net/gold_reviews_metrics";

In [0]:
CREATE TABLE IF NOT EXISTS catalog_au.golden.gold_music_segments (
    itunes_genre STRING,
    popularity_category STRING,
    is_hit STRING,
    total_tracks LONG,
    avg_popularity DOUBLE,
    avg_duration_minutes DOUBLE
)
USING DELTA
LOCATION "abfss://golden@${storageName}.dfs.core.windows.net/gold_music_segments";